# Embedding

Mit dem Embedding wandeln wir die Chunks in Vektoren um und speichern die Daten im Chunk.

- Model: deepset/gbert-large
- Input: products_chunked.jsonl
- Output: products_embedded.jsonl

Die Texte und die Specs müssen getrennt verarbeitet werden bzw. in zwei Arrays abgelegt werden, die dann per Index wieder zusammengeführt werden. Da die Eingangsdaten bereits als Chunks strukturiert sind, genügt es die zu vektorisieren Texte zu entnehmen und die Embeddings danach wieder hinzuzufügen.

Es werden alle Daten an die DB übergeben und in 16er-Schritten encodet. Progressbar ist for fun, Normalisieren ist Standard bei ChromaDB, glaube ich. Da wir die Daten nur für die Ähnlichkeitssuche benötigen ist die Länge und die darin enthaltene semantische Bedeutung nicht relevant.

In [ ]:
import json
import random
import numpy as np

from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('deepset/gbert-large')

## Daten vorbereiten

In [ ]:
products_chunked = []

with open('../data/processed/products_chunked.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        products_chunked.append(json.loads(line))

# Text only
texts = [chunk['document'] for chunk in products_chunked]

# print(texts)

## Embedding

In [ ]:
embeddings = model.encode(
    texts,
    batch_size = 16,
    show_progress_bar = True,
    normalize_embeddings = True,
    convert_to_numpy = True
)

## Zusammenfassen

In [ ]:
for chunk, embedding in zip (products_chunked, embeddings):
    chunk['embedding'] = embedding.tolist()

with open('../data/processed/products_embedded.jsonl', 'w', encoding='utf-8') as f:
    for chunk in products_chunked:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

## Evaluieren der Embeddings

In [ ]:
# Längenvergleich
assert len(embeddings) == len(products_chunked)

# Stichproben
sample_idx = random.sample(range(len(embeddings)), int(len(embeddings) * 0.01))
#sample_idx = [0]

for idx in sample_idx:
    text = products_chunked[idx]['document']
    embd = embeddings[idx]
    norm = np.linalg.norm(embd)
    test = model.encode([text], normalize_embeddings=True)[0]
    similarity = np.dot(embd, test)

    print(f"Index: {idx}")
    print(f"Text: {text[:60]}...")
    print(f"Shape: {embd}, Norm: {norm:.4f}")
    print(f"Similarity: {similarity:.8f}")

assert not np.any(np.isnan(embeddings))
assert not np.any(np.isinf(embeddings))

print(f"Shape: {embeddings.shape}")
print(f"Dtype: {embeddings.dtype}")